# Notebook 05 — Stage 1B: best-source analysis fields

**Role in the pipeline:** picks the *best available* value per row for
each chemistry variable (TA, observed pH, calculated pH, pCO₂, DIC),
records *which source* each value came from, classifies sample / CRM /
standard rows, and produces the analysis-ready subset that Stage 2 reads.

```text
04_stage1a.ipynb
   └── <stage1a_out>/data/staged.csv
                       │
                       ▼
          05_stage1b.ipynb            ◄── THIS NOTEBOOK
            • coalesce TA/pH/pCO2/DIC across precedence lists
            • record per-row provenance (which column won)
            • normalise QC status, TA units, pH scale
            • classify is_sample_row
            • two analysis gates: safe_for_analysis_qc / _strict
            • write analysis_fields.csv (everything) +
              analysis_ready_samples.csv (sample-only subset)
                                  │
                                  ▼
                       06_stage2.ipynb → ... → 08_stage4
```

**What's new vs Stage 1A**

Stage 1A's alias resolution picks **one column** per canonical name and
copies the whole thing. Stage 1B's coalescing picks **one value per
row** by walking a precedence list — and records which source won, so
the lineage is auditable per measurement.

This is the SQL `COALESCE(...)` / PySpark `coalesce(...)` pattern. The
new bit is the row-level source-tracking; see
`05_stage1b.README.md` §7 for the design rationale.


## Parameters

Single tagged `parameters` cell. The default `INPUT_CSV` and `OUT_DIR`
are intentionally `None`. This prevents accidental runs against old local
files. In normal use, `run_pipeline.sh` or Papermill supplies these paths.


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
# Required. The runner passes:
#   INPUT_CSV = <stage1a_out>/data/staged.csv
#   OUT_DIR   = <stage1b_out>
#
# Keep these as None so an interactive run fails clearly instead of
# accidentally using an old local path.
INPUT_CSV = None
OUT_DIR = None

# --- Config override (optional) ------------------------------------
# Deep-merges onto the union of roap.schema.DEFAULT_CONFIG and
# roap.stage1b.STAGE1B_DEFAULTS. Useful to change precedence lists
# or the safe-for-analysis policy without editing code.
CONFIG_PATH = None

# --- Stage 1B behaviour --------------------------------------------
# If True, skip Parquet writes (CSV only).
NO_PARQUET = False

# Useful during development: prepare everything but write nothing.
DRY_RUN = False

# If True, print preview tables in console form instead of display().
PRINT_COLUMNS = False


## Setup

The Stage lives in `roap.stage1b`: `run(df, stage_config)` builds the
best-source fields, normalises QC status, units and scales, adds the
presence, scale and range flags and returns a `StageResult` without
touching disk. This notebook reads the input, calls `run`, writes the
result and shows previews.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from roap import stage1b
from roap.common import as_bool, die, is_none_like
from roap.stage_config import load_stage_config
from roap.stage_result import write_stage_result

## Load input CSV and config

In [ ]:
NO_PARQUET = as_bool(NO_PARQUET)
DRY_RUN = as_bool(DRY_RUN)
PRINT_COLUMNS = as_bool(PRINT_COLUMNS)

if is_none_like(INPUT_CSV):
    die("INPUT_CSV is required. Run through run_pipeline.sh or set INPUT_CSV.")
if is_none_like(OUT_DIR):
    die("OUT_DIR is required. Run through run_pipeline.sh or set OUT_DIR.")
config_path = None if is_none_like(CONFIG_PATH) else str(CONFIG_PATH).strip()

src_csv = Path(str(INPUT_CSV).strip()).expanduser().resolve()
if not src_csv.exists():
    die(f"File not found: {src_csv}\nDid Notebook 04 (Stage 1A) run successfully? Stage 1B reads its staged.csv.")
if src_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {src_csv.name}")
out_root = Path(str(OUT_DIR).strip()).expanduser().resolve()

# Stage 1B defaults <- this Stage's optional config file (#41).
stage_cfg = load_stage_config("stage1b", config_path)
for _warning in stage_cfg.warnings:
    print(f"WARNING: {_warning}")

df_in = pd.read_csv(src_csv)
print(f"Source CSV    : {src_csv}")
print(f"Rows          : {len(df_in):,}")
print(f"Config source : {stage_cfg.source or 'DEFAULT_CONFIG + STAGE1B_DEFAULTS'}")

## Run Stage 1B

For each variable family `stage1b.run` walks the precedence list and picks
the first non-NA value per row, writing the value (`ta_best_umolkg`,
`ph_best`, `ph_co2sys`, `pco2_best_uatm`, `dic_best_umol_kg`) and the
column it came from (`ta_best_source`, ...). It then normalises QC status,
TA units and pH scales, adds provenance, presence, scale and range flags
(range flags read the **best** fields), and builds the Sample-only subset
with two gates:

- `safe_for_analysis_qc`: neither `ta_qc_status` nor `ph_qc_status` is FAIL
  (and, under `phstd_fail_blocks_corrected_ph`, no corrected pH from a
  failed pH standard).
- `safe_for_analysis_strict`: `safe_for_analysis_qc` and no range, unit,
  scale or core-chemistry-missing flag.

In [ ]:
try:
    result = stage1b.run(df_in, stage_cfg, source=str(src_csv))
except ValueError as exc:
    die(str(exc))

result.manifest["output_root"] = str(out_root)
result.manifest["parameters"].update(
    {
        "INPUT_CSV": str(src_csv),
        "OUT_DIR": str(out_root),
        "CONFIG_PATH": CONFIG_PATH,
        "config_path_resolved": config_path,
        "NO_PARQUET": NO_PARQUET,
        "DRY_RUN": DRY_RUN,
        "PRINT_COLUMNS": PRINT_COLUMNS,
    }
)
df = result.data["data/analysis_fields.csv"]
df_ready = result.data["data/analysis_ready_samples.csv"]
missing_tbl = result.tables["logs/missingness.csv"]
sources = result.manifest["field_sources"]
print("TA precedence used :", sources["ta_precedence_used"])
print("pH precedence used :", sources["ph_precedence_used"])
print(f"Sample rows        : {result.manifest['row_counts']['sample_rows']}")
print(f"Analysis-ready     : {len(df_ready):,}")
print(f"  safe_for_analysis_qc     : {result.manifest['row_counts']['rows_safe_for_analysis_qc']}")
print(f"  safe_for_analysis_strict : {result.manifest['row_counts']['rows_safe_for_analysis_strict']}")

## Quick previews

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "station_id", "depth_m",
        "ta_best_umolkg", "ta_best_source",
        "ph_best", "ph_best_source",
        "ph_co2sys",
        "pco2_best_uatm", "dic_best_umol_kg",
        "ta_units_normalized",
        "ph_scale_observed_normalized",
        "pressure_output_dbar",
        "safe_for_analysis_qc", "safe_for_analysis_strict",
    ]
    if c in df.columns or c in df_ready.columns
]

if PRINT_COLUMNS:
    print(missing_tbl.head(20).to_string(index=False))
    print("\n=== Analysis fields preview ===")
    print(df[[c for c in preview_cols if c in df.columns]].head(10).to_string(index=False))
    print("\n=== Analysis-ready preview ===")
    print(df_ready[[c for c in preview_cols if c in df_ready.columns]].head(10).to_string(index=False))
elif display is not None:
    display(missing_tbl.head(15))
    display(df[[c for c in preview_cols if c in df.columns]].head(10))
    display(df_ready[[c for c in preview_cols if c in df_ready.columns]].head(10))


## Write outputs

```
<OUT_DIR>/
    data/analysis_fields.csv          (and .parquet)   # all rows + best fields
    data/analysis_ready_samples.csv   (and .parquet)   # ◄── Stage 2 input
    reports/report.md
    logs/manifest.json, effective_config.json, missingness.csv
    run_bundle_manifest.json
```

`DRY_RUN = True` skips the writes. `NO_PARQUET = True` writes CSV only.
The run bundle hashes the input, the config and every file this run
wrote, so `roap verify-bundle <OUT_DIR>` can check them later.

In [ ]:
if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    written = write_stage_result(result, out_root, parquet=not NO_PARQUET)
    print(f"Stage 1B complete: {len(written)} files under {out_root}")
    print(f"  -> Stage 2 input: {written['data/analysis_ready_samples.csv']}")